# LumenY — Model Training v3 (Sub-hourly Timeframes)

**Architecture:** 5 quantile models × 3 horizons = 15 LightGBM models total.

**Horizons:** 5min, 15min, 1H (sub-hourly focus)

**Quantiles:** Q10, Q25, Q50, Q75, Q90

**Output per inference:** 5 return percentiles → direction probability, expected move, cone boundaries.

**Validation:** Walk-forward on training set only. Final model trained up to TRAIN_END cutoff.

**Key difference from v2:** Targets sub-hourly price structure (5min, 15min, 1H). Uses sub-hourly features parquet.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from sklearn.metrics import mean_pinball_loss

FEATURES_DIR = Path("../backend/data/features")
MODELS_DIR   = Path("../backend/models_3")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS       = ["5min", "15min", "1H"]
QUANTILES      = [0.10, 0.25, 0.50, 0.75, 0.90]
QUANTILE_NAMES = ["Q10", "Q25", "Q50", "Q75", "Q90"]

# ── CRITICAL: Training cutoff ──
# The final deployed model will ONLY see data up to this date.
# Everything after is truly unseen — for honest evaluation only.
TRAIN_END = "2024-06-30"

print("Ready.")
print(f"Total models to train: {len(HORIZONS) * len(QUANTILES)}")
print(f"Training cutoff: {TRAIN_END} (data after this is NEVER seen by the model)")


## 1. Load Dataset

In [ ]:
df = pd.read_parquet(FEATURES_DIR / "all_pairs_features_labels_subhourly.parquet")

label_cols   = [f"label_{h}" for h in HORIZONS]
drop_cols    = label_cols + ["pair"]
feature_cols = [c for c in df.columns if c not in drop_cols]

# Split into train and test
df_train = df[df.index <= TRAIN_END]
df_test  = df[df.index > TRAIN_END]

print(f"Full dataset:  {df.shape}  ({df.index.min().date()} -> {df.index.max().date()})")
print(f"Features:      {len(feature_cols)}")
print(f"Train set:     {len(df_train):,} rows  ({df_train.index.min().date()} -> {df_train.index.max().date()})")
print(f"Test set:      {len(df_test):,} rows  ({df_test.index.min().date()} -> {df_test.index.max().date()})")
print(f"Pairs:         {df['pair'].unique()}")
print(f"Label cols:    {label_cols}")


## 2. Walk-Forward Splits

In [ ]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

# Walk-forward splits on TRAINING data only (nothing after TRAIN_END)
splits = walk_forward_splits(len(df_train))
print(f'Walk-forward splits: {len(splits)} (all within training set, before {TRAIN_END})')
for i, (train_idx, test_idx) in enumerate(splits):
    print(f'  Fold {i+1}: Train {df_train.index[train_idx[0]].date()} to {df_train.index[train_idx[-1]].date()} | Test {df_train.index[test_idx[0]].date()} to {df_train.index[test_idx[-1]].date()} ({len(test_idx):,} rows)')

## 3. LightGBM Quantile Config

In [ ]:
def get_lgbm_params(quantile):
    return {
        'objective':         'quantile',
        'alpha':             quantile,
        'metric':            'quantile',
        'boosting_type':     'gbdt',
        'n_estimators':      5000,
        'learning_rate':     0.02,
        'num_leaves':        64,
        'max_depth':         6,
        'min_child_samples': 50,
        'feature_fraction':  0.7,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
        'device': 'gpu',
    }

print('Params ready. objective=quantile with pinball loss.')

## 4. Train All 15 Models


In [ ]:
all_results = {}
X_all = df_train[feature_cols]  # ONLY training data

for horizon in HORIZONS:
    print(f'\n{"="*55}')
    print(f'HORIZON: {horizon}')
    print(f'{"="*55}')

    y_all = df_train[f'label_{horizon}']
    valid_mask = y_all.notna() 
    X_clean = X_all[valid_mask].ffill().bfill()
    y_clean = y_all[valid_mask]
    print(f'Training samples: {len(X_clean):,} (all before {TRAIN_END})')

    horizon_results = {}
    splits = walk_forward_splits(len(X_clean))
    oof_preds  = {q: np.full(len(X_clean), np.nan) for q in QUANTILES}
    best_iters = {q: [] for q in QUANTILES}

    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        print(f'\n  Training {q_name} (alpha={q})...')
        params = get_lgbm_params(q)
        fold_pinball = []

        for fold, (train_idx, test_idx) in enumerate(splits):
            X_train = X_clean.iloc[train_idx]
            y_train = y_clean.iloc[train_idx]
            X_test  = X_clean.iloc[test_idx]
            y_test  = y_clean.iloc[test_idx]

            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_train, y_train,
                eval_set=[(X_test, y_test)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
            )

            preds = model.predict(X_test)
            oof_preds[q][test_idx] = preds
            fold_pinball.append(mean_pinball_loss(y_test, preds, alpha=q))
            best_iters[q].append(model.best_iteration_)

        mean_pb = np.mean(fold_pinball)
        print(f'    CV Pinball: {mean_pb:.6f} | Best iters: {best_iters[q]}')

        # Train final model on ALL training data (but NOTHING after TRAIN_END)
        avg_iter = max(50, int(np.mean(best_iters[q])))
        final_model = lgb.LGBMRegressor(**{**params, 'n_estimators': avg_iter})
        final_model.fit(X_clean, y_clean)

        q_int = int(q * 100)
        joblib.dump({'model': final_model, 'quantile': q, 'horizon': horizon, 'feature_cols': feature_cols},
                    MODELS_DIR / f'model_{horizon}_Q{q_int}.joblib')

        horizon_results[q_name] = {'pinball': mean_pb}

    # Quantile coverage (OOF — within training set only)
    print(f'\n  Quantile Coverage (OOF):')
    print(f'  {"Quantile":<10} {"Target":<10} {"Actual":<10} {"Gap"}')
    print(f'  {"-"*42}')

    valid_oof = ~np.isnan(oof_preds[QUANTILES[0]])
    y_oof = y_clean.values[valid_oof]

    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        preds_oof = oof_preds[q][valid_oof]
        coverage  = np.mean(y_oof <= preds_oof)
        gap       = coverage - q
        flag      = 'OK' if abs(gap) < 0.03 else 'NEEDS CALIBRATION'
        print(f'  {q_name:<10} {q:<10.2f} {coverage:<10.3f} {gap:+.3f}  {flag}')
        horizon_results[q_name]['coverage'] = coverage

    all_results[horizon] = horizon_results

print(f'\nAll {len(HORIZONS) * len(QUANTILES)} models trained and saved to {MODELS_DIR}!')
print(f'Models trained on data up to {TRAIN_END} ONLY.')

## 5. Calibration — Coverage Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor("#080c14")

for ax, horizon in zip(axes.flatten(), HORIZONS):
    ax.set_facecolor("#080c14")
    targets = QUANTILES
    actuals = [all_results[horizon][q]["coverage"] for q in QUANTILE_NAMES]

    ax.plot([0,1],[0,1],"--",color=(1,1,1,0.2),linewidth=1,label="Perfect")
    ax.plot(targets, actuals, "o-", color="#4fc3f7", linewidth=2, markersize=8, label="Model")

    for t, a, name in zip(targets, actuals, QUANTILE_NAMES):
        ax.annotate(f"{name}\n({a:.2f})", (t, a), textcoords="offset points",
                    xytext=(8,0), color="white", fontsize=7)

    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel("Target quantile", color="white")
    ax.set_ylabel("Actual coverage", color="white")
    ax.set_title(f"{horizon} Quantile Coverage", color="white")
    ax.tick_params(colors="white")
    ax.legend(facecolor="#1a2332", labelcolor="white", fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor("#1a2332")

    mce = np.mean(np.abs(np.array(actuals) - np.array(targets)))
    ax.text(0.05, 0.92, f"MCE: {mce:.3f}", transform=ax.transAxes, color="#4fc3f7", fontsize=9)

plt.suptitle("Quantile Coverage Calibration — v3 Sub-hourly Models", color="white", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Feature Importance (Q50 models)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.patch.set_facecolor("#080c14")

for ax, horizon in zip(axes.flatten(), HORIZONS):
    bundle = joblib.load(MODELS_DIR / f"model_{horizon}_Q50.joblib")
    importance = pd.Series(bundle["model"].feature_importances_, index=feature_cols)
    importance = importance.sort_values(ascending=True).tail(20)

    ax.barh(importance.index, importance.values, color="#4fc3f7", alpha=0.8)
    ax.set_facecolor("#080c14")
    ax.tick_params(colors="white", labelsize=8)
    ax.set_title(f"{horizon} Q50 Top 20 Features", color="white")
    for spine in ax.spines.values(): spine.set_edgecolor("#1a2332")

plt.suptitle("Feature Importance — Q50 models (v3)", color="white", fontsize=14)
plt.tight_layout()
plt.show()


## 7. Inference Test

In [ ]:
def predict_quantiles(pair, horizon, df_features):
    pair_data = df_features[df_features["pair"] == pair][feature_cols].ffill().bfill()
    if len(pair_data) == 0: return None
    latest = pair_data.iloc[[-1]]

    q_preds = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f"model_{horizon}_Q{int(q*100)}.joblib")
        q_preds[q_name] = float(bundle["model"].predict(latest)[0])

    q50  = q_preds["Q50"]
    qs   = np.array(QUANTILES)
    vals = np.array([q_preds[n] for n in QUANTILE_NAMES])

    if np.all(vals > 0):   p_down = 0.05
    elif np.all(vals < 0): p_down = 0.95
    else:                  p_down = float(np.interp(0, vals, qs))

    p_up = 1 - p_down
    direction = "bearish" if p_down > p_up else "bullish"
    probability = max(p_down, p_up)
    if probability < 0.55: direction = "neutral"

    return {
        "pair": pair, "horizon": horizon,
        "direction": direction, "probability": round(probability, 4),
        "p_up": round(p_up, 4), "p_down": round(p_down, 4),
        "expected_move_pct": round(q50 * 100, 4),
        "quantiles": {n: round(q_preds[n]*100, 4) for n in QUANTILE_NAMES},
        "cone": {
            "inner":  [round(q_preds["Q25"]*100,4), round(q_preds["Q75"]*100,4)],
            "outer":  [round(q_preds["Q10"]*100,4), round(q_preds["Q90"]*100,4)],
            "center": round(q50*100, 4),
        }
    }


print(f"{chr(39)}{chr(39)}{chr(39)}")
print(f"{"Pair":<10} {"H":<8} {"Dir":<10} {"Prob":<8} {"Q50":<10} {"50% cone":<28} {"80% cone"}")
print("-" * 90)
for pair in ["EURUSD", "GBPUSD", "USDJPY"]:
    for horizon in HORIZONS:
        r = predict_quantiles(pair, horizon, df)
        if r:
            c50 = f"[{r["cone"]["inner"][0]:+.3f}%, {r["cone"]["inner"][1]:+.3f}%]"
            c80 = f"[{r["cone"]["outer"][0]:+.3f}%, {r["cone"]["outer"][1]:+.3f}%]"
            print(f"{pair:<10} {horizon:<8} {r["direction"].upper():<10} {r["probability"]*100:.1f}%   {r["expected_move_pct"]:+.3f}%   {c50:<28} {c80}")


In [ ]:
# Visualize quantile distribution for one pair at 5min horizon
result = predict_quantiles("EURUSD", "5min", df)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor("#080c14")
ax.set_facecolor("#080c14")

qs   = list(result["quantiles"].keys())
vals = list(result["quantiles"].values())
colors = ["#ff4757" if v < 0 else "#4fc3f7" for v in vals]

ax.bar(qs, vals, color=colors, alpha=0.8, width=0.5)
ax.axhline(0, color=(1,1,1,0.3), linewidth=1, linestyle="--")

for i, (q, v) in enumerate(zip(qs, vals)):
    ax.text(i, v + (0.001 if v >= 0 else -0.001), f"{v:+.3f}%",
            ha="center", va="bottom" if v >= 0 else "top", color="white", fontsize=9)

ax.set_title("EURUSD 5min — Predicted Return Distribution", color="white", pad=12)
ax.set_ylabel("Predicted return (%)", color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_edgecolor("#1a2332")
ax.text(0.02, 0.95,
        f"Direction: {result["direction"].upper()}  |  P: {result["probability"]*100:.1f}%  |  Expected: {result["expected_move_pct"]:+.3f}%",
        transform=ax.transAxes, color="#4fc3f7", fontsize=9, va="top")
plt.tight_layout()
plt.show()


In [ ]:
print("=" * 55)
print("TRAINING COMPLETE — v3 Sub-hourly Models")
print("=" * 55)
print(f"Models saved: {len(list(MODELS_DIR.glob('*.joblib')))} files")
print(f"Location: {MODELS_DIR.resolve()}")
print(f"Horizons: {HORIZONS}")
print(f"Train cutoff: {TRAIN_END}")
print(f"\nFiles:")
for f in sorted(MODELS_DIR.glob("*.joblib")):
    print(f"  {f.name:<35} {f.stat().st_size/1024/1024:.1f} MB")